# Ultravox v0.5 (Llama-3.2-1B)

In [1]:
import os
import tempfile
from pathlib import Path

import librosa
import soundfile as sf
import numpy as np
import pandas as pd
import transformers
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score

/root/autodl-tmp/env/ultravox/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
!source /etc/network_turbo

设置成功
注意：仅限于学术用途和加速访问github/huggingface，不承诺稳定性保证


In [3]:
CACHE_DIR    = "/root/autodl-tmp/LLM_Model"
PROJECT_ROOT = Path("/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection")
MODEL_ID     = "fixie-ai/ultravox-v0_5-llama-3_2-1b"

os.environ["HF_HOME"] = CACHE_DIR

pipe = transformers.pipeline(
    model=MODEL_ID,
    trust_remote_code=True,
    model_kwargs={"cache_dir": CACHE_DIR},
)

print("Ultravox model loaded.")

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use cuda:0


Ultravox model loaded.


In [4]:
SYSTEM_PROMPT = (
    "You are a clinical speech-language pathologist specialized in detecting "
    "Alzheimer's disease and dementia from spontaneous speech. You analyze speech "
    "patterns including: word-finding difficulties, semantic paraphasias, empty speech, "
    "reduced syntactic complexity, repetitions, incomplete utterances, and pragmatic "
    "impairments. Based on the audio, classify the speaker."
)

USER_PROMPT = (
    "Listen to this speech sample carefully. Based on the speech characteristics, "
    "is this speaker showing signs of dementia or is this a healthy control? "
    "Answer with exactly one word: 'Dementia' or 'Control'."
)

In [5]:
TMP_WAV_DIR = Path(tempfile.mkdtemp(prefix="ultravox_wav_"))


def ensure_wav(audio_path: Path) -> Path:
    """Convert mp3 to 16kHz mono wav via librosa if needed."""
    if audio_path.suffix.lower() == ".wav":
        return audio_path
    wav_path = TMP_WAV_DIR / f"{audio_path.stem}.wav"
    if not wav_path.exists():
        audio, sr = librosa.load(str(audio_path), sr=16000, mono=True)
        sf.write(str(wav_path), audio, sr)
    return wav_path

In [6]:
VALID_LABELS = {"Dementia", "Control"}


def classify_audio(wav_path: Path) -> str:
    """Classify a single audio file. Returns raw model response."""
    audio, sr = librosa.load(str(wav_path), sr=16000, mono=True)

    turns = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"<|audio|>\n{USER_PROMPT}"},
    ]

    output = pipe(
        {"audio": audio, "turns": turns, "sampling_rate": sr},
        max_new_tokens=64,
    )
    return output

In [7]:
OUTPUT_DIR = PROJECT_ROOT / "results" / "ultravox"


def parse_prediction(raw: str) -> str | None:
    """Extract prediction from model output via keyword matching."""
    text = raw.lower()
    has_dementia = "dementia" in text
    has_control  = "control" in text or "healthy" in text
    if has_dementia and not has_control:
        return "Dementia"
    if has_control and not has_dementia:
        return "Control"
    return None


def evaluate_dataset(csv_path, audio_dir, name=""):
    df = pd.read_csv(csv_path)
    label_map = {0: "Control", 1: "Dementia"}
    predictions, skipped = [], 0

    audio_dir = Path(audio_dir)
    print(f"[{name}] audio_dir={audio_dir}, exists={audio_dir.exists()}")

    for idx, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc=name)):
        label_dir = label_map[row["ad"]]
        matches = list(audio_dir.glob(f"{label_dir}/{row['session_id']}.*"))
        if not matches:
            skipped += 1
            continue
        try:
            raw = classify_audio(ensure_wav(matches[0]))
            pred = parse_prediction(raw)
        except Exception as e:
            raw, pred = str(e), None
        if idx < 3:
            print(f"  DEBUG [{idx}] session={row['session_id']} raw={repr(raw[:200])} pred={pred}")
        if pred is None:
            print(f"  INVALID [{idx}] session={row['session_id']} true={label_dir} raw={repr(raw[:300])}")
        predictions.append({"session_id": row["session_id"], "true": label_dir, "pred": pred, "raw": raw})

    # Save predictions to CSV
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    out_csv = OUTPUT_DIR / f"{name}.csv"
    pd.DataFrame(predictions).to_csv(out_csv, index=False)
    print(f"  Saved to {out_csv}")

    valid = [p for p in predictions if p["pred"] is not None]
    y_true = [p["true"] for p in valid]
    y_pred = [p["pred"] for p in valid]
    n, total = len(valid), len(df)
    ctrl = [p for p in valid if p["true"] == "Control"]
    dem  = [p for p in valid if p["true"] == "Dementia"]

    print(f"[{name}]")
    print(f"  Accuracy:    {accuracy_score(y_true, y_pred):.4f}")
    print(f"  F1:          {f1_score(y_true, y_pred, pos_label='Dementia'):.4f}")
    print(f"  Control Acc: {sum(p['pred']=='Control'  for p in ctrl)/max(len(ctrl),1):.4f}")
    print(f"  Dementia Acc:{sum(p['pred']=='Dementia' for p in dem) /max(len(dem),1) :.4f}")
    print(f"  Valid: {n}/{total}  Skipped: {skipped}")

In [8]:
import sys; sys.path.insert(0, str(PROJECT_ROOT / "train"))
from data_split import create_test_csv

csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Pitt", "Pitt", "Pitt_xlsr_features", xlsr=True)
    
audio_dir = PROJECT_ROOT / "data/raw/Pitt"
evaluate_dataset(csv, audio_dir, "Pitt-raw")

[Pitt-raw] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/raw/Pitt, exists=True


Pitt-raw:   0%|          | 2/551 [00:01<06:36,  1.38it/s]

  DEBUG [0] session=002-0 raw='Control.' pred=Control
  DEBUG [1] session=002-1 raw='Control.' pred=Control


Pitt-raw:   1%|          | 4/551 [00:01<03:07,  2.92it/s]

  DEBUG [2] session=002-2 raw='Control.' pred=Control


Pitt-raw: 100%|██████████| 551/551 [01:13<00:00,  7.49it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/results/ultravox/Pitt-raw.csv
[Pitt-raw]
  Accuracy:    0.5027
  F1:          0.2751
  Control Acc: 0.9298
  Dementia Acc:0.1683
  Valid: 551/551  Skipped: 0


In [9]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Lu", "Lu", "Lu_xlsr_features", xlsr=True)
    
audio_dir = PROJECT_ROOT / "data/raw/Lu"
evaluate_dataset(csv, audio_dir, "Lu-raw")

[Lu-raw] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/raw/Lu, exists=True


Lu-raw:   1%|▏         | 1/74 [00:00<00:10,  7.09it/s]

  DEBUG [0] session=F22_000 raw='Healthy control.' pred=Control


Lu-raw:   3%|▎         | 2/74 [00:00<00:08,  8.01it/s]

  DEBUG [1] session=F22_001 raw='Control.' pred=Control


...

  INVALID [57] session=F18_000 true=Dementia raw='I cannot provide a diagnosis for someone who has not been tested by a medical professional. If you or someone you know is concerned about cognitive decline, I encourage seeking help from a qualified healthcare provider. Is there anything else I can help you with?'


Lu-raw:  88%|████████▊ | 65/74 [00:12<00:02,  3.90it/s]

  INVALID [63] session=F24_000 true=Dementia raw='I cannot provide a diagnosis for someone who is not speaking with me. If you or someone you know is concerned about their health, I encourage seeking help from a qualified healthcare professional. Is there anything else I can help you with?'


Lu-raw: 100%|██████████| 74/74 [00:13<00:00,  5.41it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/results/ultravox/Lu-raw.csv
[Lu-raw]
  Accuracy:    0.6143
  F1:          0.4255
  Control Acc: 0.9706
  Dementia Acc:0.2778
  Valid: 70/74  Skipped: 0


In [10]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Demucs"
evaluate_dataset(csv, audio_dir, "Pitt-Demucs")

[Pitt-Demucs] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-Demucs, exists=True


Pitt-Demucs:   0%|          | 1/551 [00:00<01:05,  8.35it/s]

  DEBUG [0] session=002-0 raw='Control.' pred=Control


Pitt-Demucs:   0%|          | 2/551 [00:00<01:11,  7.63it/s]

  DEBUG [1] session=002-1 raw='Dementia.' pred=Dementia


...

  DEBUG [2] session=002-2 raw='Control.' pred=Control


Pitt-Demucs:  98%|█████████▊| 538/551 [01:08<00:01,  7.85it/s]

  INVALID [536] session=672-0 true=Dementia raw='I cannot provide a diagnosis for someone who has not been tested by a professional. Can I help you with anything else?'


Pitt-Demucs: 100%|██████████| 551/551 [01:09<00:00,  7.91it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/results/ultravox/Pitt-Demucs.csv
[Pitt-Demucs]
  Accuracy:    0.4873
  F1:          0.2210
  Control Acc: 0.9421
  Dementia Acc:0.1299
  Valid: 550/551  Skipped: 0


In [11]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Demucs"
evaluate_dataset(csv, audio_dir, "Lu-Demucs")

[Lu-Demucs] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-Demucs, exists=True


Lu-Demucs:   1%|▏         | 1/74 [00:00<00:07,  9.35it/s]

  DEBUG [0] session=F22_000 raw='Healthy control.' pred=Control
  DEBUG [1] session=F22_001 raw='Control.' pred=Control


Lu-Demucs:   4%|▍         | 3/74 [00:00<00:07, 10.14it/s]

  DEBUG [2] session=F26_000 raw='Control.' pred=Control


...

  INVALID [61] session=F21_000 true=Dementia raw="I'm done.\n\nThe picture shows an elderly woman sitting alone in her house, looking at something up for her. She seems confused and disoriented, as indicated by her gaze towards the unknown object. The tone of her voice is hesitant and uncertain, suggesting that she may be experiencing some cognitive "


Lu-Demucs:  88%|████████▊ | 65/74 [00:09<00:01,  4.69it/s]

  INVALID [63] session=F24_000 true=Dementia raw='I cannot provide a diagnosis for someone who is not speaking with me. If you or someone you know is concerned about their health, I encourage seeking help from a qualified healthcare professional. Is there anything else I can help you with?'


Lu-Demucs: 100%|██████████| 74/74 [00:10<00:00,  7.19it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/results/ultravox/Lu-Demucs.csv
[Lu-Demucs]
  Accuracy:    0.5857
  F1:          0.3556
  Control Acc: 0.9706
  Dementia Acc:0.2222
  Valid: 70/74  Skipped: 0


In [12]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Denoiser"
evaluate_dataset(csv, audio_dir, "Pitt-Denoiser")

[Pitt-Denoiser] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-Denoiser, exists=True


Pitt-Denoiser:   0%|          | 1/551 [00:00<00:55,  9.84it/s]

  DEBUG [0] session=002-0 raw='Control.' pred=Control


Pitt-Denoiser:   0%|          | 2/551 [00:00<00:59,  9.23it/s]

  DEBUG [1] session=002-1 raw='Dementia.' pred=Dementia


...

  DEBUG [2] session=002-2 raw='Control.' pred=Control


Pitt-Denoiser:  98%|█████████▊| 538/551 [00:57<00:01,  8.73it/s]

  INVALID [536] session=672-0 true=Dementia raw='I cannot provide a diagnosis for someone who has not been tested by a professional. Can I help you with anything else?'


Pitt-Denoiser: 100%|██████████| 551/551 [00:58<00:00,  9.46it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/results/ultravox/Pitt-Denoiser.csv
[Pitt-Denoiser]
  Accuracy:    0.5036
  F1:          0.2946
  Control Acc: 0.9091
  Dementia Acc:0.1851
  Valid: 550/551  Skipped: 0


In [13]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Denoiser"
evaluate_dataset(csv, audio_dir, "Lu-Denoiser")

[Lu-Denoiser] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-Denoiser, exists=True


Lu-Denoiser:   3%|▎         | 2/74 [00:00<00:06, 11.98it/s]

  DEBUG [0] session=F22_000 raw='Healthy control.' pred=Control
  DEBUG [1] session=F22_001 raw='Control.' pred=Control
  DEBUG [2] session=F26_000 raw='Dementia.' pred=Dementia


Lu-Denoiser:  41%|████      | 30/74 [00:02<00:05,  8.64it/s]

  INVALID [27] session=F49_000 true=Control raw='I cannot provide a diagnosis for someone who has not been evaluated by a qualified healthcare professional. If you or someone you know is concerned about cognitive decline, I encourage seeking help from a medical professional. Is there anything else I can help you with?'


...

  INVALID [57] session=F18_000 true=Dementia raw='I cannot provide a diagnosis for someone who has not been tested by a medical professional. If you or someone you know is concerned about cognitive decline, I encourage seeking help from a qualified healthcare provider. Is there anything else I can help you with?'


Lu-Denoiser:  86%|████████▋ | 64/74 [00:06<00:01,  6.62it/s]

  INVALID [61] session=F21_000 true=Dementia raw="I'm not capable of describing images or visual content.\n\nHowever, I can provide an analysis of the speech characteristics:\n\nThe speaker appears to be speaking at a normal pace, with no noticeable slowing down or speeding up. The tone is neutral, without any obvious emotional distress or anxiety.\n\nUp"


Lu-Denoiser: 100%|██████████| 74/74 [00:07<00:00,  9.81it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/results/ultravox/Lu-Denoiser.csv
[Lu-Denoiser]
  Accuracy:    0.5571
  F1:          0.2791
  Control Acc: 0.9706
  Dementia Acc:0.1667
  Valid: 70/74  Skipped: 0


In [14]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-FRCRN_SE"
evaluate_dataset(csv, audio_dir, "Pitt-FRCRN_SE")

[Pitt-FRCRN_SE] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-FRCRN_SE, exists=True


Pitt-FRCRN_SE:   0%|          | 1/551 [00:00<00:55,  9.83it/s]

  DEBUG [0] session=002-0 raw='Control.' pred=Control


Pitt-FRCRN_SE:   0%|          | 2/551 [00:00<00:57,  9.51it/s]

  DEBUG [1] session=002-1 raw='Control.' pred=Control


...

  DEBUG [2] session=002-2 raw='Control.' pred=Control


Pitt-FRCRN_SE:  98%|█████████▊| 538/551 [00:57<00:01,  8.72it/s]

  INVALID [536] session=672-0 true=Dementia raw='I cannot provide a diagnosis for someone who has not been tested by a professional. Can I help you with anything else?'


Pitt-FRCRN_SE: 100%|██████████| 551/551 [00:58<00:00,  9.40it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/results/ultravox/Pitt-FRCRN_SE.csv
[Pitt-FRCRN_SE]
  Accuracy:    0.4945
  F1:          0.2446
  Control Acc: 0.9380
  Dementia Acc:0.1461
  Valid: 550/551  Skipped: 0


In [15]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-FRCRN_SE"
evaluate_dataset(csv, audio_dir, "Lu-FRCRN_SE")

[Lu-FRCRN_SE] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-FRCRN_SE, exists=True


Lu-FRCRN_SE:   4%|▍         | 3/74 [00:00<00:09,  7.82it/s]

  DEBUG [0] session=F22_000 raw='I cannot provide a diagnosis for someone who has not been tested by a professional. Can I help you with anything else?' pred=None
  INVALID [0] session=F22_000 true=Control raw='I cannot provide a diagnosis for someone who has not been tested by a professional. Can I help you with anything else?'
  DEBUG [1] session=F22_001 raw='Control.' pred=Control
  DEBUG [2] session=F26_000 raw='Control.' pred=Control


Lu-FRCRN_SE:  41%|████      | 30/74 [00:03<00:05,  8.47it/s]

  INVALID [27] session=F49_000 true=Control raw='I cannot provide a diagnosis for someone who has not been evaluated by a qualified healthcare professional. If you or someone you know is concerned about cognitive health, I encourage seeking help from a medical professional. Is there anything else I can help you with?'


...

  INVALID [61] session=F21_000 true=Dementia raw="I'm done.\n\nThe picture shows an elderly woman sitting alone in her living room, looking at something on a table next to her. She seems lost in thought, staring at it for a long time before finally focusing on someone else. The conversation appears to be about her home, specifically mentioning that s"


Lu-FRCRN_SE:  89%|████████▉ | 66/74 [00:08<00:01,  6.68it/s]

  INVALID [63] session=F24_000 true=Dementia raw='I cannot provide a diagnosis for someone who has not been evaluated by a qualified healthcare professional. Can I help you with anything else?'


Lu-FRCRN_SE: 100%|██████████| 74/74 [00:09<00:00,  7.86it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/results/ultravox/Lu-FRCRN_SE.csv
[Lu-FRCRN_SE]
  Accuracy:    0.5147
  F1:          0.1081
  Control Acc: 0.9706
  Dementia Acc:0.0588
  Valid: 68/74  Skipped: 0


In [16]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-MossFormer"
evaluate_dataset(csv, audio_dir, "Pitt-MossFormer")

[Pitt-MossFormer] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-MossFormer, exists=True


Pitt-MossFormer:   0%|          | 1/551 [00:00<00:56,  9.71it/s]

  DEBUG [0] session=002-0 raw='Control.' pred=Control


Pitt-MossFormer:   0%|          | 2/551 [00:00<01:01,  8.91it/s]

  DEBUG [1] session=002-1 raw='Dementia.' pred=Dementia


...

  DEBUG [2] session=002-2 raw='Control.' pred=Control


Pitt-MossFormer:  98%|█████████▊| 538/551 [00:57<00:01,  8.27it/s]

  INVALID [536] session=672-0 true=Dementia raw='I cannot provide a diagnosis for someone who has not been tested by a medical professional. Can I help you with anything else?'


Pitt-MossFormer: 100%|██████████| 551/551 [00:58<00:00,  9.36it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/results/ultravox/Pitt-MossFormer.csv
[Pitt-MossFormer]
  Accuracy:    0.4945
  F1:          0.2527
  Control Acc: 0.9298
  Dementia Acc:0.1526
  Valid: 550/551  Skipped: 0


In [17]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-MossFormer"
evaluate_dataset(csv, audio_dir, "Lu-MossFormer")

[Lu-MossFormer] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-MossFormer, exists=True


Lu-MossFormer:   4%|▍         | 3/74 [00:00<00:09,  7.82it/s]

  DEBUG [0] session=F22_000 raw='I cannot provide a diagnosis for someone who has not been tested by a professional. Can I help you with anything else?' pred=None
  INVALID [0] session=F22_000 true=Control raw='I cannot provide a diagnosis for someone who has not been tested by a professional. Can I help you with anything else?'
  DEBUG [1] session=F22_001 raw='Control.' pred=Control
  DEBUG [2] session=F26_000 raw='Control.' pred=Control


Lu-MossFormer:   5%|▌         | 4/74 [00:01<00:20,  3.48it/s]

  INVALID [3] session=F29_000 true=Control raw='I don\'t see anything else.\n\nThe speech characteristics I observed include:\n\n- Word-finding difficulties (e.g., "Yeah", "right at")\n- Semantic paraphasias (e.g., "cabinet" instead of "cabinets")\n- Empty speech (e.g., "Yeah", "Yeah")\n-'


...

  INVALID [61] session=F21_000 true=Dementia raw="I'm done.\n\nThe picture shows an elderly woman sitting alone in her living room, looking at something she has placed up for her. She seems distracted and not fully engaged in conversation. The background noise suggests that there may be other people present, but they are not visible.\n\nBased on the sp"


Lu-MossFormer:  89%|████████▉ | 66/74 [00:08<00:01,  6.68it/s]

  INVALID [63] session=F24_000 true=Dementia raw='I cannot provide a diagnosis for someone who has not been evaluated by a qualified healthcare professional. Can I help you with anything else?'


Lu-MossFormer: 100%|██████████| 74/74 [00:09<00:00,  8.03it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/results/ultravox/Lu-MossFormer.csv
[Lu-MossFormer]
  Accuracy:    0.5147
  F1:          0.1951
  Control Acc: 0.9688
  Dementia Acc:0.1111
  Valid: 68/74  Skipped: 0


In [18]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Resemble"
evaluate_dataset(csv, audio_dir, "Pitt-Resemble")

[Pitt-Resemble] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-Resemble, exists=True


Pitt-Resemble:   0%|          | 1/551 [00:00<01:06,  8.31it/s]

  DEBUG [0] session=002-0 raw='Control.' pred=Control


Pitt-Resemble:   0%|          | 2/551 [00:00<01:08,  8.03it/s]

  DEBUG [1] session=002-1 raw='Control.' pred=Control


...

  DEBUG [2] session=002-2 raw='Control.' pred=Control


Pitt-Resemble:  96%|█████████▌| 527/551 [01:08<00:03,  6.39it/s]

  INVALID [525] session=636-0 true=Dementia raw='I cannot provide a diagnosis for someone who has not been tested by a professional. Can I help you with anything else?'


Pitt-Resemble: 100%|██████████| 551/551 [01:12<00:00,  7.63it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/results/ultravox/Pitt-Resemble.csv
[Pitt-Resemble]
  Accuracy:    0.5145
  F1:          0.2955
  Control Acc: 0.9380
  Dementia Acc:0.1818
  Valid: 550/551  Skipped: 0


In [19]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Resemble"
evaluate_dataset(csv, audio_dir, "Lu-Resemble")

[Lu-Resemble] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-Resemble, exists=True


Lu-Resemble:   1%|▏         | 1/74 [00:00<00:07,  9.36it/s]

  DEBUG [0] session=F22_000 raw='Healthy control.' pred=Control
  DEBUG [1] session=F22_001 raw='Control.' pred=Control


Lu-Resemble:   4%|▍         | 3/74 [00:00<00:06, 10.18it/s]

  DEBUG [2] session=F26_000 raw='Control.' pred=Control


...

  INVALID [47] session=F11_000 true=Dementia raw="I know I don't see anything."


Lu-Resemble:  89%|████████▉ | 66/74 [00:09<00:01,  6.31it/s]

  INVALID [63] session=F24_000 true=Dementia raw='I cannot provide a diagnosis for someone who has not been evaluated by a qualified healthcare professional. Can I help you with anything else?'


Lu-Resemble: 100%|██████████| 74/74 [00:09<00:00,  7.56it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/results/ultravox/Lu-Resemble.csv
[Lu-Resemble]
  Accuracy:    0.5362
  F1:          0.2727
  Control Acc: 0.9394
  Dementia Acc:0.1667
  Valid: 69/74  Skipped: 0
